# CSV → Parquet Conversion & Batch Processing
Chuyển đổi file CSV lớn sang Parquet, merge lại, và xử lý theo batch cho Kaggle (30GB RAM, 19.2GB output)

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os
from pathlib import Path
import gc
import psutil

# Setup paths
INPUT_DIR = '/kaggle/input/datasets/seyhed'
OUTPUT_DIR = '/kaggle/working'
PARQUET_DIR = os.path.join(OUTPUT_DIR, 'parquet_files')
MERGED_DIR = os.path.join(OUTPUT_DIR, 'merged_data')

# Create directories
os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# Dataset configuration
DATASETS = {
    'nf-unsw-nb15-v3': '/kaggle/input/datasets/seyhed/nf-unsw-nb15-v3/NF-UNSW-NB15-v3.csv',
    'nf-ton-iot-v3': '/kaggle/input/datasets/seyhed/nf-ton-iot-v3/NF-ToN-IoT-v3.csv',
    'nf-cicids2018-v3': '/kaggle/input/datasets/seyhed/nf-cicids2018-v3/NF-CICIDS2018-v3.csv',
    'nf-bot-iot-v3': '/kaggle/input/datasets/seyhed/nf-bot-iot-v3/NF-BoT-IoT-v3.csv'
}

# Constants
BATCH_SIZE = 100000  # Xử lý 100K dòng mỗi lần

def get_memory_usage():
    """Lấy dung lượng RAM đang sử dụng (GB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

def csv_to_parquet_chunked(csv_path, output_name, chunk_rows=BATCH_SIZE):
    """Chuyển CSV sang Parquet theo batch để tiết kiệm RAM"""
    print(f"\n{'='*60}")
    print(f"Converting {output_name}...")
    print(f"{'='*60}")
    
    chunk_files = []
    chunk_num = 0
    
    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_rows)):
        mem_usage = get_memory_usage()
        print(f"Processing chunk {i+1}... Memory: {mem_usage:.2f}GB / 30GB", end='\r')
        
        # Lưu thành Parquet
        parquet_path = os.path.join(PARQUET_DIR, f'{output_name}_chunk_{chunk_num}.parquet')
        chunk.to_parquet(parquet_path, compression='snappy', index=False)
        chunk_files.append(parquet_path)
        chunk_num += 1
        
        del chunk
        gc.collect()
    
    print(f"\n✓ Converted {output_name}: {chunk_num} chunks")
    return chunk_files

def merge_parquet_files(parquet_files, output_path):
    """Merge nhiều file parquet thành 1 file"""
    print(f"\nMerging {len(parquet_files)} parquet files...")
    
    dfs = []
    for i, pf in enumerate(parquet_files):
        mem_usage = get_memory_usage()
        print(f"Reading file {i+1}/{len(parquet_files)}... Memory: {mem_usage:.2f}GB / 30GB", end='\r')
        df = pd.read_parquet(pf)
        dfs.append(df)
    
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_parquet(output_path, compression='snappy', index=False)
    print(f"\n✓ Merged: {output_path}")
    print(f"  Shape: {merged_df.shape} | Size: {os.path.getsize(output_path) / (1024**3):.2f}GB")
    
    del merged_df, dfs
    gc.collect()

# STEP 1: Convert CSV to Parquet
print("\n" + "="*60)
print("STEP 1: Converting CSV files to Parquet")
print("="*60)

parquet_collections = {}
for dataset_name, csv_path in DATASETS.items():
    if os.path.exists(csv_path):
        parquet_files = csv_to_parquet_chunked(csv_path, dataset_name)
        parquet_collections[dataset_name] = parquet_files
    else:
        print(f"⚠ File not found: {csv_path}")

# STEP 2: Merge Parquet files
print("\n" + "="*60)
print("STEP 2: Merging Parquet files by dataset")
print("="*60)

merged_files = {}
for dataset_name, parquet_files in parquet_collections.items():
    merged_path = os.path.join(MERGED_DIR, f'{dataset_name}_merged.parquet')
    merge_parquet_files(parquet_files, merged_path)
    merged_files[dataset_name] = merged_path

# Clean up
import shutil
shutil.rmtree(PARQUET_DIR, ignore_errors=True)

# Summary
print("\n" + "="*60)
print("✓ CONVERSION COMPLETED")
print("="*60)

total_size = 0
for dataset_name, parquet_path in sorted(merged_files.items()):
    size_gb = os.path.getsize(parquet_path) / (1024**3)
    rows = len(pd.read_parquet(parquet_path))
    print(f"{dataset_name}: {rows:,} rows | {size_gb:.2f}GB")
    total_size += size_gb

print(f"\nTotal: {total_size:.2f}GB / 19.2GB")
print(f"Memory: {get_memory_usage():.2f}GB / 30GB")

## Batch Processing - Single Dataset

In [ ]:
def load_data_in_batches(parquet_path, batch_size=100000):
    """Load parquet file in batches"""
    df = pd.read_parquet(parquet_path)
    for i in range(0, len(df), batch_size):
        yield df.iloc[i:i+batch_size].reset_index(drop=True)

# Example
print("Processing first dataset in batches...\n")
batch_num = 0
for batch_df in load_data_in_batches(list(merged_files.values())[0]):
    batch_num += 1
    mem = get_memory_usage()
    print(f"Batch {batch_num}: {len(batch_df):,} rows | Memory: {mem:.2f}GB")
    # YOUR CODE HERE
    if batch_num >= 3:
        print("...")
        break
    del batch_df
    gc.collect()

## Batch Processing - All Datasets Combined

In [ ]:
def load_all_datasets_in_batches(files_dict, batch_size=100000):
    """Load all datasets combined in batches"""
    batch_num = 0
    for dataset_name, parquet_path in files_dict.items():
        for batch_df in load_data_in_batches(parquet_path, batch_size):
            batch_num += 1
            mem = get_memory_usage()
            print(f"Batch {batch_num}: {len(batch_df):,} rows | {dataset_name} | Memory: {mem:.2f}GB")
            # YOUR CODE HERE
            del batch_df
            gc.collect()
    return batch_num

# Uncomment to run:
# total_batches = load_all_datasets_in_batches(merged_files)
# print(f"\nTotal batches: {total_batches}")

print("✓ Batch processing templates ready")

## File Summary & Memory Report

In [ ]:
print("\n" + "="*60)
print("FILE SUMMARY & MEMORY REPORT")
print("="*60)

print(f"\nOutput Location: {MERGED_DIR}\n")

total_size = 0
for dataset_name, parquet_path in sorted(merged_files.items()):
    if os.path.exists(parquet_path):
        size_gb = os.path.getsize(parquet_path) / (1024**3)
        df = pd.read_parquet(parquet_path)
        print(f"{dataset_name}:")
        print(f"  Rows: {len(df):,} | Columns: {len(df.columns)}")
        print(f"  Size: {size_gb:.2f}GB")
        print()
        total_size += size_gb
        del df

print(f"Total Output Size: {total_size:.2f}GB / 19.2GB")
print(f"Remaining Storage: {19.2 - total_size:.2f}GB")
print(f"\nCurrent Memory: {get_memory_usage():.2f}GB / 30GB")
print(f"Status: {'✅ OK' if get_memory_usage() < 30 else '⚠️  WARNING - High memory usage'}")

print("\n" + "="*60)
print("✓ Ready for processing!")
print("="*60)